# 001 Deep Agents Overview

这是 Deep Agents 学习线的第一份 Notebook。

配套官方文档：

- [Deep Agents Overview](https://docs.langchain.com/oss/python/deepagents/overview)
- [Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)

本课不连接 Tavily，但会连接真实的大模型网关。第一课做两件事：

1. 建立 Deep Agents 的心智模型。
2. 验证大模型网关连通性。

学习目标：

1. 理解 Deep Agents 是什么，以及它能解决什么问题。
2. 区分普通 Agent 和 Deep Agent 的能力差异。
3. 理解 Deep Agents 的核心组件：规划、工具、子代理、文件系统。
4. 读取项目 `.env` 配置并验证大模型网关连通性。
5. 明确后续课程每一步要产出什么。

## 0. 检查依赖

后续课程需要这些依赖：

```text
deepagents     -> Deep Agents 框架
tavily-python  -> Tavily 搜索 API
langchain      -> LangChain 基础库
openai         -> 调用 OpenAI-compatible 模型网关
python-dotenv  -> 读取 .env 配置
```

第一课先检查环境。

In [1]:
import importlib.metadata

try:
    print('deepagents', importlib.metadata.version('deepagents'))
except importlib.metadata.PackageNotFoundError:
    print('deepagents: not installed')

try:
    print('tavily-python', importlib.metadata.version('tavily-python'))
except importlib.metadata.PackageNotFoundError:
    print('tavily-python: not installed')

try:
    print('langchain', importlib.metadata.version('langchain'))
except importlib.metadata.PackageNotFoundError:
    print('langchain: not installed')

try:
    print('openai', importlib.metadata.version('openai'))
except importlib.metadata.PackageNotFoundError:
    print('openai: not installed')

try:
    print('python-dotenv', importlib.metadata.version('python-dotenv'))
except importlib.metadata.PackageNotFoundError:
    print('python-dotenv: not installed')

deepagents 0.6.8
tavily-python 0.7.26
langchain 1.3.7
openai 2.36.0
python-dotenv 1.1.1


## 1. Deep Agents 是什么

Deep Agents 是 LangChain 提供的一个高级 Agent 框架。

它和普通 Agent 最大的区别是：

```text
普通 Agent：接收问题 -> 调用工具 -> 返回答案
Deep Agent：接收复杂任务 -> 自动规划 -> 调用工具 -> 委派子任务 -> 管理上下文 -> 综合输出
```

如果用 Java 后端类比：

```text
普通 Agent 像一个简单的 Service 方法。
Deep Agent 像一个带任务编排、子服务委派、上下文管理的完整业务流程。
```

Deep Agents 自动具备以下能力：

1. **任务规划**：自动把复杂任务拆成多个子任务（todo list）
2. **工具调用**：调用你注册的工具（如搜索、文件读写）
3. **子代理委派**：把子任务交给专门的子代理
4. **上下文管理**：用文件系统工具管理大上下文，避免超出模型限制
5. **结果综合**：把所有子任务结果综合成最终答案

In [2]:
agent_comparison = {
    '普通 Agent': {
        '输入': '用户问题',
        '处理': '调用工具',
        '输出': '直接回答',
        '适合场景': '简单问答、单步工具调用',
    },
    'Deep Agent': {
        '输入': '复杂任务',
        '处理': '规划 -> 工具调用 -> 子代理委派 -> 上下文管理 -> 综合',
        '输出': '结构化报告',
        '适合场景': '研究报告、复杂分析、多步骤任务',
    },
}

for agent_type, capabilities in agent_comparison.items():
    print(f'=== {agent_type} ===')
    for key, value in capabilities.items():
        print(f'  {key}: {value}')
    print()

=== 普通 Agent ===
  输入: 用户问题
  处理: 调用工具
  输出: 直接回答
  适合场景: 简单问答、单步工具调用

=== Deep Agent ===
  输入: 复杂任务
  处理: 规划 -> 工具调用 -> 子代理委派 -> 上下文管理 -> 综合
  输出: 结构化报告
  适合场景: 研究报告、复杂分析、多步骤任务



## 2. Deep Agents 的核心组件

Deep Agents 由以下组件构成：

| 组件 | 作用 | 示例 |
|---|---|---|
| Model | 大模型，负责推理和决策 | OpenAI / Anthropic / Google |
| Tools | 工具，供 agent 调用 | 搜索、文件读写 |
| System Prompt | 系统提示，指导 agent 行为 | 角色设定、工具使用说明 |
| write_todos | 内置工具，用于任务规划 | 自动拆分任务 |
| write_file / read_file | 内置文件系统 | 管理大上下文 |
| Subagents | 子代理，处理专门任务 | 研究子代理、验证子代理 |

推荐边界：

```text
Model 负责推理和决策。
Tools 是 agent 的手和脚。
System Prompt 是 agent 的大脑设定。
write_todos 是 agent 的计划本。
文件系统是 agent 的草稿纸。
Subagents 是 agent 的助手。
```

In [3]:
components = {
    'Model': '大模型，负责推理和决策',
    'Tools': '工具，供 agent 调用（如搜索、文件读写）',
    'System Prompt': '系统提示，指导 agent 行为',
    'write_todos': '内置工具，用于任务规划',
    'write_file / read_file': '内置文件系统，管理大上下文',
    'Subagents': '子代理，处理专门任务',
}

for name, role in components.items():
    print(f'[{name}] {role}')

[Model] 大模型，负责推理和决策
[Tools] 工具，供 agent 调用（如搜索、文件读写）
[System Prompt] 系统提示，指导 agent 行为
[write_todos] 内置工具，用于任务规划
[write_file / read_file] 内置文件系统，管理大上下文
[Subagents] 子代理，处理专门任务


## 3. 先用小数据模拟 Deep Agent 执行流程

假设用户给 agent 一个任务：

```text
写一份关于 LangGraph 的研究报告
```

Deep Agent 会自动执行以下步骤：

```text
1. 规划：拆成多个子任务
2. 研究：调用搜索工具收集信息
3. 存储：把搜索结果写入文件
4. 综合：基于所有信息写报告
```

这里用 Python 数据结构模拟这个过程。

In [4]:
from pprint import pprint

# 用户任务
task = '写一份关于 LangGraph 的研究报告'
print('用户任务:', task)
print('-' * 80)

# 1. 规划阶段：agent 自动拆分任务
todos = [
    {'id': 1, 'task': '搜索 LangGraph 基本概念和架构', 'status': 'pending'},
    {'id': 2, 'task': '搜索 LangGraph 核心功能和应用场景', 'status': 'pending'},
    {'id': 3, 'task': '搜索 LangGraph 与其他框架的对比', 'status': 'pending'},
    {'id': 4, 'task': '基于搜索结果撰写研究报告', 'status': 'pending'},
]
print('1. 规划阶段 - 自动拆分任务:')
pprint(todos)
print('-' * 80)

# 2. 执行阶段：调用工具 + 存储结果
search_results = [
    {
        'todo_id': 1,
        'query': 'LangGraph 基本概念',
        'results': ['LangGraph 是 LangChain 的状态图框架...', '它支持循环、分支和人类在环...'],
    },
    {
        'todo_id': 2,
        'query': 'LangGraph 应用场景',
        'results': ['适用于复杂 Agent 流程编排...', '支持多轮对话和状态管理...'],
    },
]
print('2. 执行阶段 - 搜索并存储:')
pprint(search_results)
print('-' * 80)

# 3. 综合阶段：撰写报告
report = (
    '# LangGraph 研究报告\n\n'
    '## 1. 概述\n'
    'LangGraph 是 LangChain 的状态图框架，支持循环、分支和人类在环。\n\n'
    '## 2. 应用场景\n'
    '适用于复杂 Agent 流程编排，支持多轮对话和状态管理。\n\n'
    '## 3. 总结\n'
    'LangGraph 提供了强大的 Agent 流程编排能力。\n'
)
print('3. 综合阶段 - 撰写报告:')
print(report)

用户任务: 写一份关于 LangGraph 的研究报告
--------------------------------------------------------------------------------
1. 规划阶段 - 自动拆分任务:
[{'id': 1, 'status': 'pending', 'task': '搜索 LangGraph 基本概念和架构'},
 {'id': 2, 'status': 'pending', 'task': '搜索 LangGraph 核心功能和应用场景'},
 {'id': 3, 'status': 'pending', 'task': '搜索 LangGraph 与其他框架的对比'},
 {'id': 4, 'status': 'pending', 'task': '基于搜索结果撰写研究报告'}]
--------------------------------------------------------------------------------
2. 执行阶段 - 搜索并存储:
[{'query': 'LangGraph 基本概念',
  'results': ['LangGraph 是 LangChain 的状态图框架...', '它支持循环、分支和人类在环...'],
  'todo_id': 1},
 {'query': 'LangGraph 应用场景',
  'results': ['适用于复杂 Agent 流程编排...', '支持多轮对话和状态管理...'],
  'todo_id': 2}]
--------------------------------------------------------------------------------
3. 综合阶段 - 撰写报告:
# LangGraph 研究报告

## 1. 概述
LangGraph 是 LangChain 的状态图框架，支持循环、分支和人类在环。

## 2. 应用场景
适用于复杂 Agent 流程编排，支持多轮对话和状态管理。

## 3. 总结
LangGraph 提供了强大的 Agent 流程编排能力。



## 4. 为什么需要 Deep Agents

如果用户问一个简单问题，普通 Agent 就够了。

但如果任务是：

```text
写一份关于 X 的研究报告，需要涵盖：
- 基本概念
- 核心功能
- 应用场景
- 与其他方案的对比
- 未来展望
```

普通 Agent 会：
- 尝试一次调用搜索所有信息
- 上下文容易超出限制
- 回答质量不稳定

Deep Agent 会：
- 自动拆分任务
- 分步搜索并存储中间结果
- 用文件系统管理大上下文
- 必要时委派子代理
- 最终综合成结构化报告

所以 Deep Agents 不是替代普通 Agent，而是处理更复杂任务的升级方案。

In [5]:
use_cases = {
    '简单问答': {
        '示例': '今天天气怎么样？',
        '推荐': '普通 Agent',
        '原因': '单步工具调用即可',
    },
    '单步工具调用': {
        '示例': '帮我查一下北京到上海的机票',
        '推荐': '普通 Agent',
        '原因': '调用一个 API 就能解决',
    },
    '研究报告': {
        '示例': '写一份关于 LangGraph 的研究报告',
        '推荐': 'Deep Agent',
        '原因': '需要多步搜索、上下文管理、综合输出',
    },
    '复杂分析': {
        '示例': '分析某技术的优缺点、应用场景、与其他方案对比',
        '推荐': 'Deep Agent',
        '原因': '需要多维度信息收集和综合分析',
    },
}

for use_case, info in use_cases.items():
    print(f'场景: {use_case}')
    print(f'  示例: {info["示例"]}')
    print(f'  推荐: {info["推荐"]}')
    print(f'  原因: {info["原因"]}')
    print()

场景: 简单问答
  示例: 今天天气怎么样？
  推荐: 普通 Agent
  原因: 单步工具调用即可

场景: 单步工具调用
  示例: 帮我查一下北京到上海的机票
  推荐: 普通 Agent
  原因: 调用一个 API 就能解决

场景: 研究报告
  示例: 写一份关于 LangGraph 的研究报告
  推荐: Deep Agent
  原因: 需要多步搜索、上下文管理、综合输出

场景: 复杂分析
  示例: 分析某技术的优缺点、应用场景、与其他方案对比
  推荐: Deep Agent
  原因: 需要多维度信息收集和综合分析



## 6. 后续课程预告

下一课要做的是：

```text
安装 deepagents 和 tavily-python
配置 API Key
创建 internet_search 工具
```

再下一课：

```text
使用 create_deep_agent 创建 agent
配置 model、tools、system_prompt
```

最终目标：

```text
运行 agent 并观察执行过程
流式输出 agent 事件
```

## 7. 练习

请你思考后回答：

1. Deep Agents 和普通 Agent 最大的区别是什么？
2. Deep Agents 自动具备哪些核心能力？
3. 什么场景下应该用 Deep Agents，而不是普通 Agent？
4. 如果让你设计一个 Deep Agent 来写技术对比报告，你会给它注册哪些工具？